In [2]:
import sys
import os

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DRIVE_ROOT = '/content/drive/MyDrive/AA-STAL/data_pipeline'

    if DRIVE_ROOT not in sys.path:
        sys.path.append(DRIVE_ROOT)

    print(f"Directory impostata su: {os.getcwd()}")
except ImportError:
    # Fallback per esecuzione in locale
    DRIVE_ROOT = '.'
    print("Esecuzione in locale")

# Cambia la directory di lavoro in modo che i path relativi (es. configs/, saved_models/) funzionino
os.chdir(DRIVE_ROOT)
print(f"Directory di lavoro ripristinata su: {os.getcwd()}")

Mounted at /content/drive
Directory impostata su: /content
Directory di lavoro ripristinata su: /content/drive/MyDrive/AA-STAL/data_pipeline


In [ ]:
!pip install -U transformers accelerate bitsandbytes qwen-vl-utils torchvision

In [1]:
!pip install -U opencv-python

In [ ]:
import os
import json
import glob
import cv2

def crop_video_frames(main_dir):
    # Itera su tutte le sottocartelle della directory principale
    for folder_name in os.listdir(main_dir):
        folder_path = os.path.join(main_dir, folder_name)

        # Salta i file e processa solo le cartelle
        if not os.path.isdir(folder_path):
            continue

        print(f"Elaborazione cartella: {folder_name}")

        # Cerca il file JSON all'interno della cartella
        json_files = glob.glob(os.path.join(folder_path, "*.json"))
        if not json_files:
            print(f"-> Nessun file JSON trovato in {folder_name}. Salto.")
            continue

        json_path = json_files[0]
        with open(json_path, 'r') as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                print(f"-> Errore nel parsing del JSON in {folder_name}. Salto.")
                continue

        # Recupera e ordina alfabeticamente i frame (.jpg)
        frame_paths = sorted(glob.glob(os.path.join(folder_path, "*.jpg")))
        if not frame_paths:
            print(f"-> Nessun frame .jpg trovato in {folder_name}. Salto.")
            continue

        detected_objects = data.get("detected_objects", {})

        # Itera su ogni oggetto rilevato nel JSON (es. robot_1000, robot_1001)
        for obj_id, obj_info in detected_objects.items():
            bboxes = obj_info.get("bbox", [])

            # Crea la cartella di output per i ritagli di questo specifico oggetto
            output_dir = os.path.join(folder_path, "crops", obj_id)
            os.makedirs(output_dir, exist_ok=True)

            # Associa ogni bbox al rispettivo frame in ordine sequenziale
            for idx, bbox in enumerate(bboxes):
                # Evita crash se ci sono più bbox registrate rispetto ai frame fisici
                if idx >= len(frame_paths):
                    break

                # CONTROLLO AGGIUNTO: Salta l'iterazione se la bbox è null o non valida
                if bbox is None or len(bbox) != 4:
                    continue

                frame_path = frame_paths[idx]
                frame_name = os.path.basename(frame_path)

                # Carica l'immagine
                img = cv2.imread(frame_path)
                if img is None:
                    continue

                height, width, _ = img.shape

                # Estrazione coordinate originarie
                xmin, ymin, xmax, ymax = bbox

                # Calcolo larghezza e altezza della bounding box originaria
                box_w = xmax - xmin
                box_h = ymax - ymin

                # Calcolo del 10% di padding per lato per includere lo sfondo
                pad_w = box_w * 0.10
                pad_h = box_h * 0.10

                # Applicazione del padding e vincolo entro i limiti [0.0, 1.0]
                xmin_pad = max(0.0, xmin - pad_w)
                ymin_pad = max(0.0, ymin - pad_h)
                xmax_pad = min(1.0, xmax + pad_w)
                ymax_pad = min(1.0, ymax + pad_h)

                # Conversione da coordinate normalizzate a pixel (interi)
                x1 = int(xmin_pad * width)
                y1 = int(ymin_pad * height)
                x2 = int(xmax_pad * width)
                y2 = int(ymax_pad * height)

                # Ritaglio della regione d'interesse (ROI)
                crop = img[y1:y2, x1:x2]

                # Salva il ritaglio se valido
                if crop.size > 0:
                    crop_filename = os.path.join(output_dir, frame_name)
                    cv2.imwrite(crop_filename, crop)

    print("\nProcesso completato per tutte le cartelle.")

if __name__ == "__main__":
    # Configura qui il percorso della tua directory principale
    PATH_DIRETTORE_PRINCIPALE = "/content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT/video_general_obj_det_finished"
    crop_video_frames(PATH_DIRETTORE_PRINCIPALE)

Elaborazione cartella: robot-15_scene12
Elaborazione cartella: robot-15_scene36


In [ ]:
import json
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info



if __name__ == '__main__':
  # 1. Configurazione dizionario azioni
  # Mappa la label dell'attore al suo set specifico di azioni.
  action_sets = {
      "robotic arm": ["grasping", "moving", "idle"],
      "vehicle": ["turning", "roatating", "moving forward", "idle"],
      "person": ["walking", "sit", "stand"]
      }

  # 2. Configurazione Modello (4-bit per T4)
  quantization_config = BitsAndBytesConfig(
      load_in_4bit=True,
      bnb_4bit_compute_dtype=torch.float16,
      bnb_4bit_quant_type="nf4",
      bnb_4bit_use_double_quant=True
  )

  model_id = "Qwen/Qwen2.5-VL-7B-Instruct"

  model = AutoModelForVision2Seq.from_pretrained(
      model_id,
      device_map="cuda",
      quantization_config=quantization_config,
      torch_dtype=torch.float16
  )

  processor = AutoProcessor.from_pretrained(
      model_id,
      min_pixels=256 * 28 * 28,
      max_pixels=512 * 28 * 28
  )

  # 2. for every video
  od_finished_path = "/content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT/video_general_obj_det_finished"
  for folder in od_finished_path.os.listdir():
    json_file = [f for f in os.listdir(folder_path) if f.lower().endswith(".json")]
    json_path = os.path.join(od_finished_path, folder, json_file)
    with open(json_path, 'r') as f:
      data_track = json.load(f)

    for actor_key, actor_data in dati_track["detected_objects"].items():
      actor_label = data_track.get('class_name', 'unknown')
      bboxes = data_track.get('bbox', [])
      num_frames = len(bboxes)

      if num_frames == 0:
        continue

      # RECUPERO FRAME
      # Assumiamo che i crop siano stati salvati in cartelle nominate con l'actor_key.
      # Adatta questo percorso alla struttura reale delle tue directory.
      frame_paths = [f"percorso/ai/crop/{actor_key}/frame_{i:04d}.jpg" for i in range(num_frames)]

      # CAMPIONAMENTO TEMPORALE (Critico per evitare OOM)
      # Anche se il video ha 300 frame, ne passiamo un massimo di 60 equidistanti a Qwen.
      # Il modello capirà comunque l'azione, ma non saturerà la RAM della GPU.
      max_frames_to_qwen = 60
      step = max(1, num_frames // max_frames_to_qwen)
      sampled_frame_paths = frame_paths[::step]

      if not frame_paths:
        raise ValueError("no bbox found for actor" + actor_label)

      # Recupera le azioni per l'attore specifico (usa un set di default in caso di errore)
      available_actions = action_sets.get(actor_label, ["undefined"])
      formatted_actions = "\n".join([f"- [{action}]" for action in available_actions])

      # 3. Costruzione dinamica del prompt
      prompt = f"""Analyze the provided video frames, which show the temporal sequence of a single actor ({actor_label}) isolated from the background context.
      Based on the actor's movements, select the correct action from the following options:
      {formatted_actions}
      Strict Output Constraint:
      Respond ONLY with the exact action name enclosed in square brackets (e.g., [Action Name]). Do not include any explanations, introductory text, markdown formatting other than the brackets, or extra characters."""

      # 4. Strutturazione dell'Input
      # Passando una lista di percorsi file al campo "video", il modello li tratta come sequenza temporale
      messages = [
          {
              "role": "user",
              "content": [
                  {
                      "type": "video",
                      "video": frame_paths,
                      "max_pixels": 360 * 360,
                      },
                  {
                      "type": "text",
                      "text": prompt
                  },
              ],
          }
      ]

      text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
      image_inputs, video_inputs = process_vision_info(messages)

      inputs = processor(
          text=[text],
          images=image_inputs,
          videos=video_inputs,
          padding=True,
          return_tensors="pt",
      )

      inputs = inputs.to("cuda")

      # INFERENZA
      with torch.no_grad():
          generated_ids = model.generate(
              **inputs,
              max_new_tokens=20,
              temperature=0.1,
              do_sample=False
          )

      generated_ids_trimmed = [
          out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
      ]

      output_text = processor.batch_decode(
          generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
      )[0]

      # PULIZIA OUTPUT
      # Rimuove spazi vuoti e le parentesi quadre dall'output per avere la label pulita
      predicted_action = output_text.strip().strip("[]")
      print(f"[{actor_key}] Processed ({num_frames} frames). Predicted action: {predicted_action}")

      # AGGIORNAMENTO JSON: Array "action" parallelo all'array "bbox"
      actor_data["action"] = [predicted_action] * num_frames

      # Svuota la cache della GPU tra un attore e l'altro per sicurezza
      torch.cuda.empty_cache()

    # 5. Salvataggio del JSON finale aggiornato
    with open(json_output_path, 'w') as f:
        json.dump(dati_track, f, indent=4)

    print(f"\JSON saved in: {json_output_path}")